---
# PARTE I: BeautifulSoup — Extracción de Datos Web
---

## 1.1 Marco Teórico

### ¿Qué es BeautifulSoup?

**BeautifulSoup** es una biblioteca de Python especializada en **analizar y navegar documentos HTML y XML**. Su nombre proviene del poema *"Beautiful Soup"* de *Alicia en el País de las Maravillas*, haciendo referencia a su capacidad de trabajar con HTML desordenado o "sopa de etiquetas".

Actúa como un **intérprete inteligente**: recibe el código fuente HTML de una página web y lo convierte en un árbol de objetos Python que puede recorrerse, filtrarse y consultarse de forma sencilla, sin necesidad de procesar el texto crudo manualmente.

---

### ¿Qué es el Web Scraping?

El **Web Scraping** (también llamado *raspado web* o *extracción web*) es la técnica de **recolectar datos automáticamente desde páginas web**, simulando lo que haría un usuario al leer manualmente el contenido de un sitio.

El flujo general es:
1. Realizar una solicitud HTTP a una URL objetivo
2. Recibir el HTML de la página como texto
3. Analizar y filtrar ese HTML para localizar la información deseada
4. Almacenar los datos en un formato útil (CSV, base de datos, DataFrame, etc.)



---
## 1.2 Instalación y Verificación

Las librerías necesarias para este laboratorio son:
- `beautifulsoup4`: el analizador HTML principal
- `requests`: para realizar solicitudes HTTP a páginas web
- `pandas`: para organizar los datos en una tabla estructurada
- `lxml`: parser HTML de alta velocidad (motor interno de BeautifulSoup)


In [57]:
# Importación de librerías y verificación de versiones
import bs4
import requests
import pandas
print("Todas las librerías están instaladas y listas para usar.")

Todas las librerías están instaladas y listas para usar.


---
## 1.3 Caso Práctico: Sistema de Monitoreo de Catálogo Editorial

> **Contexto**: La empresa *LecturaMás S.A.C.* gestiona una cadena de librerías en Perú. El equipo de compras necesita revisar semanalmente los precios, disponibilidad y valoraciones del catálogo de su proveedor en línea para decidir qué libros reabastecer prioritariamente.
>
> Hacerlo manualmente tarda más de 3 horas por semana. Se encarga al área de sistemas desarrollar un script que extraiga automáticamente esa información y genere un reporte estructurado listo para análisis.

### Sitio objetivo
`books.toscrape.com` — Sitio público diseñado específicamente para practicar Web Scraping legalmente.

### ¿Qué extraeremos?
- Título del libro  
- Precio (en libras esterlinas, moneda del proveedor)  
- Valoración en estrellas  
- Disponibilidad (En stock / Agotado)

In [42]:
# ─────────────────────────────────────────────────────────────
# PASO 1: Importaciones y configuración del sistema
# ─────────────────────────────────────────────────────────────
from bs4 import BeautifulSoup   # El analizador HTML
import requests                  # Para solicitudes HTTP
import pandas as pd              # Para tablas de datos
import time                      # Para pausas entre solicitudes

# El sitio representa las valoraciones como palabras en inglés dentro de clases CSS
# Este diccionario las convierte a números para poder ordenar y filtrar
ESTRELLAS = {
    "One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5
}

# Cabecera HTTP: nos identificamos como un navegador real
# Algunos servidores rechazan solicitudes sin User-Agent
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

print("⚙️  Configuración inicial completada.")

⚙️  Configuración inicial completada.


El HTML del respaldo reproduce fielmente la estructura del sitio real. Esto es importante: BeautifulSoup trabajará **exactamente de la misma manera** ya sea con el HTML descargado de Internet o con el HTML almacenado localmente, porque lo que importa es la estructura, no el origen.

In [ ]:
# ─────────────────────────────────────────────────────────────
# PASO 2: Función principal — extrae libros de un HTML dado
# Funciona igual con HTML de Internet o con el HTML local
# ─────────────────────────────────────────────────────────────
def extraer_libros_de_html(html_texto: str) -> list:
    """
    Recibe una cadena de texto con HTML y devuelve
    una lista de diccionarios con los datos de cada libro.
    """
    # Creamos el árbol de análisis a partir del texto HTML
    # 'lxml' es el motor de análisis: rápido y tolerante con HTML malformado
    soup = BeautifulSoup(html_texto, "lxml")

    # Localizamos todos los bloques de producto en la página
    # Cada libro está encerrado en un <article class="product_pod">
    articulos = soup.find_all("article", class_="product_pod")

    libros_encontrados = []  # Lista donde acumulamos los resultados

    for articulo in articulos:

        # ── Título ───────────────────────────────────────────────────────
        # El atributo 'title' del enlace <a> contiene el título completo.
        # El texto visible suele estar truncado con "...", por eso usamos 'title'.
        titulo = articulo.find("h3").find("a")["title"]

        # ── Precio ──────────────────────────────────────────────
        # Ejemplo del HTML: £51.77
        # Algunas páginas pueden devolver: Â£51.77 debido a la codificación.
        precio_texto = articulo.find("p", class_="price_color").get_text(strip=True)

        # Eliminamos caracteres extraños y el símbolo de libra
        precio_texto = (
            precio_texto
                .replace("Â", "")
                .replace("£", "")
                .replace("$", "")
                .strip()
        )
        
        # Convertimos el texto a un número decimal
        precio = float(precio_texto)

        # ── Disponibilidad ───────────────────────────────────────────────
        # El texto puede ser "In stock" o "Out of stock"
        # Lo traducimos al español para el reporte final
        disponibilidad_raw = articulo.find("p", class_="availability").text.strip()
        disponibilidad = "En stock" if "In stock" in disponibilidad_raw else "Agotado"

        # ── Valoración en estrellas ───────────────────────────────────────
        # La valoración está codificada como una clase CSS adicional:
        # <p class="star-rating Three"> significa 3 estrellas.
        # ["class"] devuelve una lista, tomamos el índice [1] (el segundo elemento).
        clase_estrella = articulo.find("p", class_="star-rating")["class"][1]
        estrellas = ESTRELLAS.get(clase_estrella, 0)  # Convertimos texto a número

        # Guardamos todos los datos de este libro
        libros_encontrados.append({
            "Titulo":         titulo,
            "Precio_GBP":     precio,
            "Estrellas":      estrellas,
            "Disponibilidad": disponibilidad,
        })

    return libros_encontrados

print("✅ Función 'extraer_libros_de_html' definida correctamente.")

✅ Función 'extraer_libros_de_html' definida correctamente.


Esta función es el núcleo de BeautifulSoup en acción. Observemos cómo navega el árbol HTML con precisión:

- `find_all("article", class_="product_pod")` → busca todos los bloques de producto en la página
- `.find("h3").find("a")["title"]` → desciende al `<h3>`, luego al `<a>` y lee el atributo `title`
- `.find("p", class_="price_color").text` → encuentra el `<p>` con esa clase específica y extrae su texto
- `["class"][1]` → de la lista de clases `["star-rating", "Three"]`, toma el segundo elemento

In [ ]:
# ─────────────────────────────────────────────────────────────
# PASO 3: Obtención del HTML desde la página web
# ─────────────────────────────────────────────────────────────

URL_OBJETIVO = "https://books.toscrape.com/catalogue/page-1.html"
html_obtenido = None

print(f"🌐 Intentando conectar a: {URL_OBJETIVO}")
print()

try:

    respuesta = requests.get(
        URL_OBJETIVO,
        headers=HEADERS,
        timeout=8
    )
    
    respuesta.raise_for_status()

    html_obtenido = respuesta.text

    print(f"✅ Conexión exitosa.")
    print(f"📄 HTML recibido: {len(html_obtenido):,} caracteres")

except requests.exceptions.RequestException as e:

    print("❌ No fue posible obtener la página web.")
    print(f"Detalle del error: {e}")

    raise

🌐 Intentando conectar a: https://books.toscrape.com/catalogue/page-1.html

✅ Conexión exitosa.
📄 HTML recibido: 50,469 caracteres


In [ ]:
# ─────────────────────────────────────────────────────────────
# PASO 4: Aplicamos BeautifulSoup al HTML obtenido
# ─────────────────────────────────────────────────────────────

# Llamamos a nuestra función de extracción con el HTML disponible
lista_libros = extraer_libros_de_html(html_obtenido)

print(f"   Libros extraídos del HTML: {len(lista_libros)}")
print()

print("Vista previa de los primeros 3 registros extraídos:")
print("-" * 55)
for libro in lista_libros[:3]:
    print(f"  Título : {libro['Titulo']}")
    print(f"  Precio : £{libro['Precio_GBP']}")
    print(f"  Estrellas: {libro['Estrellas']} | Disponibilidad: {libro['Disponibilidad']}")
    print()

   Libros extraídos del HTML: 20

Vista previa de los primeros 3 registros extraídos:
-------------------------------------------------------
  Título : A Light in the Attic
  Precio : £51.77
  Estrellas: 3 | Disponibilidad: En stock

  Título : Tipping the Velvet
  Precio : £53.74
  Estrellas: 1 | Disponibilidad: En stock

  Título : Soumission
  Precio : £50.1
  Estrellas: 1 | Disponibilidad: En stock



In [61]:
# ─────────────────────────────────────────────────────────────
# PASO 5: Convertimos los datos a DataFrame de pandas
# ─────────────────────────────────────────────────────────────

# Creamos el DataFrame a partir de la lista de diccionarios
df = pd.DataFrame(lista_libros)

# Renombramos las columnas para el reporte final en español
df.columns = ["Título", "Precio (£)", "Valoración", "Disponibilidad"]

print("Catálogo completo:")
df

Catálogo completo:


,Título,Precio (£),Valoración,Disponibilidad
0,A Light in the Attic,51.77,3,En stock
1,Tipping the Velvet,53.74,1,En stock
2,Soumission,50.10,1,En stock
3,Sharp Objects,47.82,4,En stock
4,Sapiens: A Brief History of Humankind,54.23,5,En stock
5,The Requiem Red,22.65,1,En stock
6,The Dirty Little Secrets of Getting Your Dream...,33.34,4,En stock
7,The Coming Woman: A Novel Based on the Life of...,17.93,3,En stock
8,The Boys in the Boat: Nine Americans and Their...,22.60,4,En stock
9,The Black Maria,52.15,1,En stock


In [66]:
# ─────────────────────────────────────────────────────────────
# PASO 7: Exportar el catálogo a CSV para el equipo de compras
# ─────────────────────────────────────────────────────────────
from pathlib import Path

ruta_excel = Path("catalogo_libros.xlsx")

df.to_excel(ruta_excel, index=False)

---
# PARTE II: Cryptography — Protección de Información Sensible
---

## 2.1 Marco Teórico

### ¿Qué es el cifrado?

El **cifrado** (o encriptación) es el proceso de transformar información legible en un formato ilegible para cualquier persona que no posea la clave correcta para revertirlo. Es el pilar fundamental de la seguridad de la información moderna.

---

### Texto plano vs. Texto cifrado

| Concepto | Descripción | Ejemplo |
|----------|-------------|----------|
| **Texto plano** (*plaintext*) | Información original, legible por cualquiera | `DNI: 40123456, Saldo: S/. 12,500` |
| **Texto cifrado** (*ciphertext*) | Información transformada, ilegible sin la clave | `gAAAAABn7x2LQKs...==` |

---

### ¿Qué es una clave criptográfica?

La **clave** es un valor secreto que controla el proceso de cifrado y descifrado. Sin la clave correcta, recuperar el texto original es computacionalmente inviable. Existen dos paradigmas:
- **Cifrado simétrico**: la misma clave cifra y descifra (más rápido; es el que usa Fernet)
- **Cifrado asimétrico**: clave pública para cifrar, clave privada para descifrar (usado en SSL/TLS)

---

### ¿Cómo funciona Fernet?

**Fernet** es un esquema de cifrado simétrico **autenticado** que combina dos algoritmos:
- **AES-128-CBC**: cifra los datos (Advanced Encryption Standard con modo CBC)
- **HMAC-SHA256**: verifica que los datos no fueron alterados (firma de integridad)

Esta combinación garantiza **confidencialidad** (nadie puede leer los datos) e **integridad** (nadie puede modificarlos sin que se detecte). El flujo completo es:

```
DATOS ORIGINALES  →  [AES-128-CBC + HMAC-SHA256]  →  TOKEN FERNET (cifrado)
TOKEN FERNET      →  [Verificar HMAC → Descifrar]  →  DATOS ORIGINALES
```

---

### Aplicaciones reales del cifrado

| Contexto | Aplicación concreta |
|----------|--------------------|
| **Bases de datos** | Cifrar contraseñas, números de tarjeta, datos de identidad |
| **Mensajería** | WhatsApp, Signal y Telegram usan cifrado de extremo a extremo |
| **Almacenamiento** | BitLocker y FileVault cifran discos duros completos |
| **APIs y web** | Tokens JWT y sesiones en servicios web |
| **Salud** | Historiales clínicos protegidos por ley en muchos países |
| **Derecho peruano** | Ley N° 29733 — Protección de Datos Personales |

---
## 2.2 Instalación y Verificación

In [49]:
# Instalación de la librería cryptography
%pip install cryptography --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [50]:
# Verificamos la versión instalada
import cryptography

print(f"✅ Cryptography versión: {cryptography.__version__}")
print()
print("La librería está instalada y lista.")

✅ Cryptography versión: 47.0.0

La librería está instalada y lista.


---
## 2.3 Caso Práctico: Sistema de Protección de Datos de Clientes

### Escenario de negocio

> **Contexto**: *Soluciones Financieras Andinas S.A.C.* almacena en sus servidores el archivo `clientes.txt` con información personal sensible: nombres completos, DNI, correos electrónicos y saldos de cuenta.
>
> El área de TI detectó que este archivo se guarda como **texto plano**. Ante una fuga de datos o acceso no autorizado al servidor, cualquier persona podría leer toda esa información. Se solicita implementar un sistema de cifrado que:
> 1. **Cifre** el archivo antes de guardarlo en disco
> 2. Almacene la **clave** de forma separada y segura
> 3. Permita **descifrar** y recuperar los datos cuando sea necesario

### Flujo del sistema
```
clientes.txt (texto plano)
       ↓  [ CIFRAR con clave.key ]
clientes.enc (ilegible sin clave)
       ↓  [ DESCIFRAR con clave.key ]
Datos recuperados (idénticos al original)
```

In [51]:
# ─────────────────────────────────────────────────────────────
# PASO 1: Importaciones del módulo de criptografía
# ─────────────────────────────────────────────────────────────
from cryptography.fernet import Fernet   # El cifrador Fernet (AES-128 + HMAC)
from pathlib import Path                  # Manejo moderno de rutas de archivos

print("✅ Módulos de criptografía importados correctamente.")

✅ Módulos de criptografía importados correctamente.


In [52]:
# ─────────────────────────────────────────────────────────────
# PASO 2: Creamos el archivo confidencial de clientes
# Este simula los datos reales que tendría la empresa en producción
# ─────────────────────────────────────────────────────────────
contenido_clientes = """=============================================================
     DATOS CONFIDENCIALES DE CLIENTES — USO INTERNO
     Soluciones Financieras Andinas S.A.C.
     Fecha: 2026-06-24  |  Clasificación: SECRETO
=============================================================

ID  | Nombre Completo             | DNI       | Correo Electrónico      | Saldo (S/.)
----|-----------------------------|-----------| ------------------------|-------------
001 | García Quispe, Rosa María   | 40123456  | rgarcia@email.com       |  12,500.00
002 | Mamani Torres, Juan Carlos  | 41987654  | jmamani@email.com       |   8,750.50
003 | Huanca Flores, Ana Lucía    | 43215678  | ahuanca@email.com       |  45,200.00
004 | Condori Paz, Pedro Alonso   | 44561234  | pcondori@email.com      |   3,100.75
005 | Apaza Rios, Carmen Elena    | 46789012  | capaza@email.com        | 120,000.00

=============================================================
Ley N° 29733 — Protección de Datos Personales del Perú.
La divulgación no autorizada está sujeta a sanciones legales.
============================================================="""

# Escribimos el archivo de texto plano en disco
ruta_original = Path("clientes.txt")
ruta_original.write_text(contenido_clientes, encoding="utf-8")

print("📄 Archivo 'clientes.txt' creado exitosamente.")
print(f"   Tamaño: {ruta_original.stat().st_size} bytes")
print()
print("🔓 Contenido ANTES del cifrado — visible para cualquiera con acceso al servidor:")
print("=" * 65)
print(contenido_clientes)

📄 Archivo 'clientes.txt' creado exitosamente.
   Tamaño: 1141 bytes

🔓 Contenido ANTES del cifrado — visible para cualquiera con acceso al servidor:
     DATOS CONFIDENCIALES DE CLIENTES — USO INTERNO
     Soluciones Financieras Andinas S.A.C.
     Fecha: 2026-06-24  |  Clasificación: SECRETO

ID  | Nombre Completo             | DNI       | Correo Electrónico      | Saldo (S/.)
----|-----------------------------|-----------| ------------------------|-------------
001 | García Quispe, Rosa María   | 40123456  | rgarcia@email.com       |  12,500.00
002 | Mamani Torres, Juan Carlos  | 41987654  | jmamani@email.com       |   8,750.50
003 | Huanca Flores, Ana Lucía    | 43215678  | ahuanca@email.com       |  45,200.00
004 | Condori Paz, Pedro Alonso   | 44561234  | pcondori@email.com      |   3,100.75
005 | Apaza Rios, Carmen Elena    | 46789012  | capaza@email.com        | 120,000.00

Ley N° 29733 — Protección de Datos Personales del Perú.
La divulgación no autorizada está sujeta a sancion

Este es el problema que debemos resolver: el archivo es completamente legible. Si alguien accede al servidor, a una copia de seguridad, o intercepta una transferencia del archivo, toda la información confidencial quedará expuesta. La solución es cifrarlo.

In [ ]:
# ─────────────────────────────────────────────────────────────
# PASO 3: Generamos la clave criptográfica
# ─────────────────────────────────────────────────────────────

# Fernet.generate_key() crea una clave aleatoria de 256 bits,
# codificada en Base64 URL-safe, usando el generador de números
# aleatorios CRIPTOGRÁFICAMENTE SEGURO del sistema operativo.
# Cada ejecución genera una clave ÚNICA e irrepetible.

clave = Fernet.generate_key()

# En producción real, la clave se almacenaría en un gestor de secretos
# (AWS Secrets Manager, HashiCorp Vault, Azure Key Vault, etc.)
# Para este ejemplo, la guardamos en un archivo separado del de datos
ruta_clave = Path("clave.key")
ruta_clave.write_bytes(clave)   # Escribimos bytes crudos, sin codificación adicional

print("🔑 Clave criptográfica generada y guardada en 'clave.key'")
print(f"   Longitud de la clave en bytes: {len(clave)}")
print(f"   Vista parcial (Base64):        {clave[:20].decode()}...{clave[-8:].decode()}")
print()
print("━" * 65)
print("⚠️  REGLA DE ORO DE LA CRIPTOGRAFÍA:")
print("   Esta clave es la ÚNICA forma de descifrar los datos.")
print("   Si se pierde, los datos cifrados serán IRRECUPERABLES.")
print("━" * 65)

🔑 Clave criptográfica generada y guardada en 'clave.key'
   Longitud de la clave en bytes: 44
   Vista parcial (Base64):        jnI_YcSU2VbMw2cWR8bJ...AruIFmE=

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
⚠️  REGLA DE ORO DE LA CRIPTOGRAFÍA:
   Esta clave es la ÚNICA forma de descifrar los datos.
   Si se pierde, los datos cifrados serán IRRECUPERABLES.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


In [54]:
# ─────────────────────────────────────────────────────────────
# PASO 4: Ciframos completamente el archivo de clientes
# ─────────────────────────────────────────────────────────────

# Creamos el objeto Fernet con la clave generada
# Este objeto concentra toda la lógica de cifrado y descifrado
cifrador = Fernet(clave)

# Leemos el contenido del archivo original como BYTES
# (Fernet trabaja con bytes, no con strings directamente)
datos_originales = ruta_original.read_bytes()

# CIFRADO: Fernet aplica AES-128-CBC y añade un HMAC-SHA256 para integridad
# También incluye un IV (vector de inicialización) aleatorio y una marca de tiempo
datos_cifrados = cifrador.encrypt(datos_originales)

# Guardamos el resultado cifrado en un archivo con extensión .enc
ruta_cifrada = Path("clientes.enc")
ruta_cifrada.write_bytes(datos_cifrados)

print("🔒 Proceso de cifrado completado.")
print()
print(f"   Archivo original : clientes.txt   → {ruta_original.stat().st_size} bytes")
print(f"   Archivo cifrado  : clientes.enc   → {ruta_cifrada.stat().st_size} bytes")
print(f"   Overhead de Fernet: ~{ruta_cifrada.stat().st_size - ruta_original.stat().st_size} bytes (IV + HMAC + metadata)")
print()
print("👁️  Contenido del archivo DESPUÉS del cifrado (ilegible sin la clave):")
print("=" * 65)

# Mostramos los primeros 180 caracteres del contenido cifrado
preview = datos_cifrados[:180].decode("ascii", errors="replace")
print(preview)
print("...  [resto del contenido igual de ilegible]")

🔒 Proceso de cifrado completado.

   Archivo original : clientes.txt   → 1141 bytes
   Archivo cifrado  : clientes.enc   → 1612 bytes
   Overhead de Fernet: ~471 bytes (IV + HMAC + metadata)

👁️  Contenido del archivo DESPUÉS del cifrado (ilegible sin la clave):
gAAAAABqPFZOUeFcAm7YNtb05J0p2iHVdkO-YvlfD8qbRi2nfa55SL3VLctpGvumgFHeF86_GjjC5urOvLui0U1W4SY6cCiQMRRPhvDxaty9jj3UnBc8xUNnElhCZ9Ay5CbaRMgEsMixpEEvrShX8-uoBj4SmAipRv4LKKRtukj2vH4rxPL_
...  [resto del contenido igual de ilegible]


El archivo `clientes.enc` es completamente ilegible. Incluso si alguien lo interceptara o lo copiara, no obtendría información útil. El overhead de ~60 bytes que aparece se debe a que Fernet incluye: el IV aleatorio (16 bytes), el HMAC de autenticación (32 bytes), la versión del protocolo y la marca de tiempo.

In [55]:
# ─────────────────────────────────────────────────────────────
# PASO 5: Descifrado — el sistema autorizado recupera los datos
# Simula un acceso posterior por parte del personal autorizado
# ─────────────────────────────────────────────────────────────

print("⏳ Simulando acceso autorizado posterior al cifrado...")
print()

# El sistema autorizado carga la clave desde el almacén de secretos
# (en este caso, el archivo clave.key; en producción sería un gestor seguro)
clave_cargada = ruta_clave.read_bytes()
print(f"🔑 Clave cargada correctamente desde 'clave.key'")

# Creamos un nuevo objeto Fernet (simula una sesión diferente del sistema)
descifrador = Fernet(clave_cargada)

# Leemos el archivo cifrado desde disco
contenido_cifrado = ruta_cifrada.read_bytes()
print(f"📂 Archivo cifrado leído: {len(contenido_cifrado)} bytes")
print()

try:
    # DESCIFRADO: Fernet primero VERIFICA el HMAC (integridad)
    # Si el archivo fue alterado o la clave es incorrecta, lanza InvalidToken.
    # Solo si el HMAC es correcto, aplica AES-128-CBC inverso para recuperar los datos.
    datos_recuperados = descifrador.decrypt(contenido_cifrado)

    # Convertimos los bytes de vuelta a texto legible
    contenido_recuperado = datos_recuperados.decode("utf-8")

    print("✅ Descifrado exitoso. Datos recuperados íntegramente.")
    print()
    print("📄 Contenido recuperado (idéntico al archivo original):")
    print("=" * 65)
    print(contenido_recuperado)

    # Verificación formal de que los datos son exactamente iguales
    print()
    if datos_recuperados == datos_originales:
        print("✅ VERIFICACIÓN: Los datos recuperados son IDÉNTICOS al original (byte a byte).")
    else:
        print("⚠️  Los datos difieren del original.")

except Exception as e:
    # Este bloque se ejecuta si la clave es incorrecta o el archivo fue manipulado
    print(f"❌ Error al descifrar: {type(e).__name__}")
    print("   Causa posible: clave incorrecta o archivo cifrado alterado.")

⏳ Simulando acceso autorizado posterior al cifrado...

🔑 Clave cargada correctamente desde 'clave.key'
📂 Archivo cifrado leído: 1612 bytes

✅ Descifrado exitoso. Datos recuperados íntegramente.

📄 Contenido recuperado (idéntico al archivo original):
     DATOS CONFIDENCIALES DE CLIENTES — USO INTERNO
     Soluciones Financieras Andinas S.A.C.
     Fecha: 2026-06-24  |  Clasificación: SECRETO

ID  | Nombre Completo             | DNI       | Correo Electrónico      | Saldo (S/.)
----|-----------------------------|-----------| ------------------------|-------------
001 | García Quispe, Rosa María   | 40123456  | rgarcia@email.com       |  12,500.00
002 | Mamani Torres, Juan Carlos  | 41987654  | jmamani@email.com       |   8,750.50
003 | Huanca Flores, Ana Lucía    | 43215678  | ahuanca@email.com       |  45,200.00
004 | Condori Paz, Pedro Alonso   | 44561234  | pcondori@email.com      |   3,100.75
005 | Apaza Rios, Carmen Elena    | 46789012  | capaza@email.com        | 120,000.00

Ley N